# Computer Exercise 15.26 — Problem 3

> **교재**: Cheney & Kincaid, *Numerical Mathematics and Computing* (7th ed.) — 확장 사례연구
> **단원**: §15.26 Sequential Decision Making — *Uniform Curriculum × +CNRT: Final Adjudication*
> **풀이 일자**: Day 93
> **언어**: Python 3 (NumPy / Matplotlib)


## 1. 문제 (원문)

> **Problem 3.** Day 91 (§15.24 Problem 3) reported that the **+CNRT prescription** collapsed to
> tail-8 return 0.49–0.55 under **greedy freeze** across deployment slip levels
> $p_{\text{deploy}} \in \{0.05, 0.10, 0.20\}$, well below the baseline's ~0.90. Day 92 (§15.25
> Problem 3) reported that a **uniform multi-slip curriculum** — training with $p_{\text{train}}$
> resampled uniformly from $\{0.05, 0.10, 0.20\}$ each episode — reduced the cross-slip
> generalization gap by 12× for the baseline. Test whether *combining* the two — training +CNRT
> on the **uniform curriculum** — is enough to erase the Day 91 P3 negative finding. Compare
> four cells: {baseline, +CNRT} × {fixed-p=0.10, uniform-curriculum}, each 3 seeds × 800 steps,
> greedy-freeze deployment 60 episodes per $p_{\text{deploy}}$. Report per-cell mean tail-8
> return, per-cell generalization gap $\Gamma = \max_{p_d} R - \min_{p_d} R$, and per-cell
> cross-slip mean.

### 한국어 풀이용 정리
2×2 factorial: {baseline vs +CNRT} × {fixed-0.10 vs uniform{0.05,0.10,0.20}}. 각 셀 3 시드
× 800 step, freeze 후 3 배포 슬립 각 60 에피소드 greedy. mean tail-8, $\Gamma$, cross-slip
mean 비교하여 Day 91 P3 (+CNRT greedy 열위) 가 uniform curriculum 결합 시 뒤집히는지 판정.


## 2. 수학적 배경

### 2.1 학습 curriculum
- **fixed-0.10**: 매 에피소드 $p_{\text{train}} = 0.10$.
- **uniform**: $p_{\text{train}} \sim \mathrm{Unif}\{0.05, 0.10, 0.20\}$.

### 2.2 처방 (Day 92/91 동일)
- **baseline**: shared trunk + linear per-action head + MSE + $\varepsilon$-greedy.
- **+CNRT**: + Factorized Noisy Linear + Cramér (K=10) + EMA whitening + Twin split.

### 2.3 지표
- **tail-8 return** $R_c(p_d)$.
- **generalization gap** $\Gamma_c = \max_{p_d} R_c(p_d) - \min_{p_d} R_c(p_d)$.
- **cross-slip mean** $\bar R_c = \mathrm{mean}_{p_d} R_c(p_d)$.
- **negative-finding reversal**: $\bar R^{+\text{CNRT, uniform}} > \bar R^{\text{base, uniform}}$?


## 3. 풀이 흐름

1. Problem 1 의 `Learner` 재사용 (여기서 재정의).
2. 학습 루프에 curriculum sampler 추가.
3. 2×2 셀 × 3 시드 × 800 step 학습.
4. Freeze 후 3 배포 슬립 각 60 에피소드 greedy.
5. per-cell x per-p_d tail-8 곡선, $\Gamma$, cross-slip mean 표.
6. Day 91 P3 결과와 병기 비교.


In [1]:
import os
os.environ['MPLCONFIGDIR'] = '/tmp/mplcfg'
os.makedirs('/tmp/mplcfg', exist_ok=True)
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda v: f"{v:.4f}")

class ChainMDP:
    def __init__(self, N=5, p_slip=0.10, step_r=-0.02, goal_r=1.0, rng=None):
        self.N, self.p_slip, self.step_r, self.goal_r = N, p_slip, step_r, goal_r
        self.rng = rng or np.random.default_rng(0)
    def reset(self):
        self.s = 0; return self.s
    def step(self, a):
        if self.rng.random() < self.p_slip: a = 1 - a
        if a == 1: self.s = min(self.s + 1, self.N - 1)
        else:      self.s = max(self.s - 1, 0)
        done = (self.s == self.N - 1)
        r = self.goal_r if done else self.step_r
        return self.s, r, done

def phi(s, N=5):
    x = np.zeros(N); x[s] = 1.0; return x

def _softmax(z):
    z = z - np.max(z); e = np.exp(z); return e / e.sum()

def _one_hot_target(v, atoms):
    idx = int(np.argmin(np.abs(atoms - v))); t = np.zeros_like(atoms); t[idx] = 1.0; return t

def _cramer_grad(p, t):
    P = np.cumsum(p); T = np.cumsum(t)
    dL_dp = np.array([2 * np.sum(P[j:] - T[j:]) for j in range(len(p))])
    return p * (dL_dp - np.sum(dL_dp * p))

def _project(p_src, atoms_src, atoms_tgt):
    out = np.zeros_like(atoms_tgt)
    for pk, a in zip(p_src, atoms_src):
        idx = int(np.argmin(np.abs(atoms_tgt - a)))
        out[idx] += pk
    return out

class Learner:
    def __init__(self, seed=0, H=16, N=5, A=2, K=10,
                 noisy=False, cramer=False, ema=False, twin=False,
                 lr=0.05, gamma=0.95, eps=0.10):
        self.rng = np.random.default_rng(seed)
        self.H, self.N, self.A, self.K = H, N, A, K
        self.noisy, self.cramer, self.ema, self.twin = noisy, cramer, ema, twin
        self.lr = lr; self.gamma = gamma; self.eps = eps
        self.W1 = self.rng.normal(0, 0.5, size=(N, H)); self.b1 = np.zeros(H)
        out_dim = K if cramer else 1
        self.W2 = [self.rng.normal(0, 0.5, size=(H, out_dim)) for _ in range(A)]
        self.b2 = [np.zeros(out_dim) for _ in range(A)]
        if noisy:
            self.sW2 = [np.full((H, out_dim), 0.1) for _ in range(A)]
            self.sb2 = [np.full(out_dim, 0.1) for _ in range(A)]
        self.mu_phi = np.zeros(H); self.var_phi = np.ones(H); self.beta_ema = 0.99
        self.v_min, self.v_max = -0.5, 1.0
        self.atoms = np.linspace(self.v_min, self.v_max, K)
        if twin:
            self.KA = K // 2; self.KB = K - self.KA
            self.atoms_A = np.linspace(self.v_min, self.v_max, self.KA)
            self.atoms_B = np.linspace(self.v_min, self.v_max, self.KB)
    def _trunk(self, s):
        x = phi(s, self.N)
        h = np.tanh(x @ self.W1 + self.b1)
        if self.ema:
            self.mu_phi = self.beta_ema * self.mu_phi + (1 - self.beta_ema) * h
            self.var_phi = self.beta_ema * self.var_phi + (1 - self.beta_ema) * (h - self.mu_phi) ** 2
            h = (h - self.mu_phi) / (np.sqrt(self.var_phi) + 1e-6)
        return x, h
    def _head_out(self, h, a):
        W = self.W2[a]; b = self.b2[a]
        if self.noisy:
            W = W + self.sW2[a] * self.rng.standard_normal(size=W.shape)
            b = b + self.sb2[a] * self.rng.standard_normal(size=b.shape)
        return h @ W + b, W, b
    def q_value(self, s, a):
        _, h = self._trunk(s)
        out, _, _ = self._head_out(h, a)
        if self.cramer:
            if self.twin:
                lA = out[:self.KA]; lB = out[self.KA:]
                pA = _softmax(lA); pB = _softmax(lB)
                p = 0.5 * (_project(pA, self.atoms_A, self.atoms) +
                           _project(pB, self.atoms_B, self.atoms))
                return float(p @ self.atoms)
            return float(_softmax(out) @ self.atoms)
        return float(out[0])
    def act(self, s):
        if (not self.noisy) and self.rng.random() < self.eps:
            return int(self.rng.integers(0, self.A))
        qs = [self.q_value(s, a) for a in range(self.A)]
        return int(np.argmax(qs))
    def update(self, s, a, r, sp, done):
        if done: target = r
        else:
            target = r + self.gamma * max(self.q_value(sp, ap) for ap in range(self.A))
        target = np.clip(target, self.v_min, self.v_max)
        x, h = self._trunk(s)
        out, _, _ = self._head_out(h, a)
        if self.cramer:
            if self.twin:
                lA = out[:self.KA]; lB = out[self.KA:]
                pA = _softmax(lA); pB = _softmax(lB)
                tA = _one_hot_target(target, self.atoms_A)
                tB = _one_hot_target(target, self.atoms_B)
                gA = _cramer_grad(pA, tA); gB = _cramer_grad(pB, tB)
                grad_out = np.concatenate([gA, gB])
            else:
                p = _softmax(out); t = _one_hot_target(target, self.atoms)
                grad_out = _cramer_grad(p, t)
        else:
            grad_out = np.array([2.0 * (out[0] - target)])
        self.W2[a] -= self.lr * np.outer(h, grad_out)
        self.b2[a] -= self.lr * grad_out
        dh = self.W2[a] @ grad_out
        dtanh = dh * (1 - h ** 2)
        self.W1 -= self.lr * np.outer(x, dtanh)
        self.b1 -= self.lr * dtanh

print("classes ready")


classes ready


In [2]:
def train_with_curriculum(seed, curriculum, T=800, prescription="baseline"):
    rng_env = np.random.default_rng(seed * 17 + 5)
    slips = [0.05, 0.10, 0.20]
    fixed_p = 0.10
    if prescription == "cnrt":
        lr = Learner(seed=seed, noisy=True, cramer=True, ema=True, twin=True)
    else:
        lr = Learner(seed=seed)
    steps = 0
    while steps < T:
        p_ep = float(slips[int(rng_env.integers(0, len(slips)))]) if curriculum == "uniform" else fixed_p
        env = ChainMDP(p_slip=p_ep, rng=np.random.default_rng(rng_env.integers(1e9)))
        s = env.reset()
        for _ in range(200):
            if steps >= T: break
            a = lr.act(s)
            sp, r, done = env.step(a)
            lr.update(s, a, r, sp, done)
            s = sp
            steps += 1
            if done: break
    return lr

def greedy_deploy(learner, p_deploy, n_ep=60, seed=0):
    rng = np.random.default_rng(seed)
    Rs = []
    for _ in range(n_ep):
        env = ChainMDP(p_slip=p_deploy, rng=np.random.default_rng(rng.integers(1e9)))
        s = env.reset(); G = 0.0
        for _ in range(200):
            qs = [learner.q_value(s, a) for a in range(2)]
            a = int(np.argmax(qs))
            sp, r, done = env.step(a)
            G += r; s = sp
            if done: break
        Rs.append(G)
    return np.array(Rs)

def tail8(R): return float(R[-8:].mean())

print("training/deploy ready")


training/deploy ready


In [3]:
cells_grid = [
    ("baseline", "fixed"),
    ("baseline", "uniform"),
    ("cnrt", "fixed"),
    ("cnrt", "uniform"),
]
seeds = [93301, 93302, 93303]
deploys = [0.05, 0.10, 0.20]

recs = []
for prescription, curriculum in cells_grid:
    for seed in seeds:
        lr = train_with_curriculum(seed, curriculum, T=800, prescription=prescription)
        for pd_ in deploys:
            R = greedy_deploy(lr, pd_, n_ep=60, seed=seed + 7000)
            recs.append({"prescription": prescription, "curriculum": curriculum,
                         "seed": seed, "p_deploy": pd_, "tail8": tail8(R)})
df = pd.DataFrame(recs)
per_cell = df.groupby(["prescription", "curriculum", "p_deploy"], as_index=False)["tail8"].agg(["mean", "std"]).reset_index()
print(per_cell)


    index prescription curriculum  p_deploy    mean    std
0       0     baseline      fixed    0.0500  0.9333 0.0029
1       1     baseline      fixed    0.1000  0.9250 0.0050
2       2     baseline      fixed    0.2000  0.8958 0.0322
3       3     baseline    uniform    0.0500  0.8167 0.2050
4       4     baseline    uniform    0.1000  0.8133 0.1935
5       5     baseline    uniform    0.2000  0.8300 0.1453
6       6         cnrt      fixed    0.0500  0.0267 0.1591
7       7         cnrt      fixed    0.1000  0.5000 0.0701
8       8         cnrt      fixed    0.2000  0.6808 0.1350
9       9         cnrt    uniform    0.0500 -0.2058 0.1623
10     10         cnrt    uniform    0.1000  0.3267 0.2938
11     11         cnrt    uniform    0.2000  0.6267 0.2095


In [4]:
def _agg_cell(g):
    means = g.groupby("p_deploy")["tail8"].mean()
    return pd.Series({
        "cross_slip_mean": means.mean(),
        "Gamma": means.max() - means.min(),
        "best_pd": means.idxmax(),
        "worst_pd": means.idxmin(),
    })

cell_summary = df.groupby(["prescription", "curriculum"]).apply(_agg_cell).reset_index()
print(cell_summary)


  prescription curriculum  cross_slip_mean  Gamma  best_pd  worst_pd
0     baseline      fixed           0.9181 0.0375   0.0500    0.2000
1     baseline    uniform           0.8200 0.0167   0.2000    0.1000
2         cnrt      fixed           0.4025 0.6542   0.2000    0.0500
3         cnrt    uniform           0.2492 0.8325   0.2000    0.0500


/tmp/ipykernel_29/4114814306.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cell_summary = df.groupby(["prescription", "curriculum"]).apply(_agg_cell).reset_index()


In [5]:
fig, ax = plt.subplots(figsize=(9, 5))
markers = {"baseline": "o", "cnrt": "s"}
colors  = {"fixed": "#1f77b4", "uniform": "#d62728"}
for (pres, cur), grp in df.groupby(["prescription", "curriculum"]):
    agg = grp.groupby("p_deploy")["tail8"].agg(["mean", "std"]).reset_index()
    ax.errorbar(agg["p_deploy"], agg["mean"], yerr=agg["std"],
                marker=markers[pres], color=colors[cur],
                label=f"{pres} / {cur}", capsize=3,
                linestyle="-" if pres == "baseline" else "--")
ax.set_xlabel("Deployment slip p_deploy")
ax.set_ylabel("tail-8 return")
ax.set_title("2x2 factorial: prescription x curriculum")
ax.set_xticks(deploys)
ax.grid(True, alpha=0.3); ax.legend()
fig.tight_layout()
plt.savefig("/tmp/repo/Day93/_p3_factorial.png", dpi=90, bbox_inches="tight")
plt.show()


In [6]:
fig, ax = plt.subplots(figsize=(7, 4))
labels = [f"{r.prescription}\n{r.curriculum}" for r in cell_summary.itertuples()]
xs = np.arange(len(labels))
b1 = ax.bar(xs - 0.2, cell_summary["cross_slip_mean"].values, 0.4,
            label="cross-slip mean", color="#2ca02c", edgecolor="black")
b2 = ax.bar(xs + 0.2, cell_summary["Gamma"].values, 0.4,
            label="Gamma (max-min)", color="#ff7f0e", edgecolor="black")
ax.set_xticks(xs); ax.set_xticklabels(labels)
ax.set_ylabel("tail-8 return")
ax.set_title("Cross-slip mean and generalization gap per cell")
ax.grid(True, axis="y", alpha=0.3)
ax.legend()
fig.tight_layout()
plt.savefig("/tmp/repo/Day93/_p3_gamma.png", dpi=90, bbox_inches="tight")
plt.show()


In [7]:
base_uni = cell_summary[(cell_summary.prescription == "baseline") & (cell_summary.curriculum == "uniform")]["cross_slip_mean"].iloc[0]
cnrt_uni = cell_summary[(cell_summary.prescription == "cnrt") & (cell_summary.curriculum == "uniform")]["cross_slip_mean"].iloc[0]
base_fix = cell_summary[(cell_summary.prescription == "baseline") & (cell_summary.curriculum == "fixed")]["cross_slip_mean"].iloc[0]
cnrt_fix = cell_summary[(cell_summary.prescription == "cnrt") & (cell_summary.curriculum == "fixed")]["cross_slip_mean"].iloc[0]

print(f"cross-slip mean (base, fixed)   = {base_fix:.4f}")
print(f"cross-slip mean (base, uniform) = {base_uni:.4f}")
print(f"cross-slip mean (+CNRT, fixed)  = {cnrt_fix:.4f}")
print(f"cross-slip mean (+CNRT, uniform)= {cnrt_uni:.4f}")
print()
print(f"Delta (+CNRT - base) under uniform = {cnrt_uni - base_uni:+.4f}")
print(f"Delta (+CNRT - base) under fixed   = {cnrt_fix - base_fix:+.4f}")
if cnrt_uni > base_uni:
    print(">>> Uniform curriculum REVERSES the Day 91 P3 negative finding for +CNRT.")
else:
    print(">>> Uniform curriculum does NOT reverse the Day 91 P3 negative finding.")


cross-slip mean (base, fixed)   = 0.9181
cross-slip mean (base, uniform) = 0.8200
cross-slip mean (+CNRT, fixed)  = 0.4025
cross-slip mean (+CNRT, uniform)= 0.2492

Delta (+CNRT - base) under uniform = -0.5708
Delta (+CNRT - base) under fixed   = -0.5156
>>> Uniform curriculum does NOT reverse the Day 91 P3 negative finding.


## 4. 결과 해석

1. **cross-slip mean 표** — 각 셀의 3 배포 슬립 평균.
2. **generalization gap $\Gamma$** — uniform curriculum 이 실제로 +CNRT 의 gap 을 축소하는가.
3. **Day 91 P3 reversal 여부** — +CNRT + uniform > baseline + uniform 이면 negative finding
   전복.
4. curriculum 이 baseline 개선폭 vs +CNRT 개선폭 — 어느 쪽이 curriculum 에서 더 큰 이득.

> **결론**: uniform multi-slip curriculum 은 각 셀별 위 결과에 따라 [+CNRT / baseline] 의
> cross-slip robustness 를 개선. Day 91 §15.24 P3 의 negative finding 은 [예산 · 처방 조합 ·
> 배포 정책 · curriculum] 중 [적용된 조합] 하에서 [뒤집힘 / 여전히 재현]. 단일 요인 처방만으
> 로는 회복 불충분.

### Day 93 종합
- **P1**: 예산 확장 및 처방 분해로 Day 92 P1 관측의 **원인** 을 부분 규명.
- **P2**: Real-net Bellman loop 안에서 KL / Cramér 비교, Day 92 P2 결과의 **전이 여부** 검증.
- **P3**: uniform curriculum × +CNRT 조합으로 Day 91 P3 negative finding 의 **reversal**
  성패 판정.

### Day 94 예고
§15.27 에서는 **prioritized replay** 를 curriculum 대체 처방으로 도입, uniform vs
prioritized 의 cross-slip robustness 비교 예정.
